In [2]:
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import os
from openpyxl import load_workbook
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.styles import numbers

month_ele = r"U:\OneDrive_2025-05-10\Dokumenty\Chytil\source\elektrina_mesic"
month_plyn = r"U:\OneDrive_2025-05-10\Dokumenty\Chytil\source\plyn_mesic"
alltime_ele = r"U:\OneDrive_2025-05-10\Dokumenty\Chytil\source\elektrina_alltime"
alltime_plyn = r"U:\OneDrive_2025-05-10\Dokumenty\Chytil\source\plyn_alltime"
ceny_chytil = r"U:\OneDrive_2025-05-10\Dokumenty\Chytil\source\ceny_chytil.xlsx"
rozsireny = r"S:\Praha\OBCHOD-INTERNÍ\Obchod-Reporting\Sam\Report_archiv\Report_rozšířený"

final_folder = r"U:\OneDrive_2025-05-10\Dokumenty\Chytil"
prev_month = (datetime.now() - relativedelta(months=1)).strftime("%m%Y")
final_filename = f"Chytil_Final_{prev_month}.xlsx"
final_path = os.path.join(final_folder, final_filename)

# create a function to read the newest file in the folder

def load_newest_excel(folder, header_row=0, dtype=None):
    # listing all the files in the folder, folder is an argument in here
    files = [f for f in os.listdir(folder) if f.endswith(('.xlsx', '.xls'))]
    
    # if the folder is empty, stop the whole operation
    if not files: 
        print(f"No excel files in {folder}")
        return None
    
    # find the newest file by modification time
    newest = max(files, key= lambda f: os.path.getmtime(os.path.join(folder, f)))
    
    full_path = os.path.join(folder, newest)
    print(f"Loading: {newest}")
    df = pd.read_excel(full_path, header=header_row, engine='calamine', dtype=dtype)
    df = df[[col for col in df.columns if 'Unnamed' not in str(col)]]
    return df

# loading the newest file from every folder — EAN/EIC forced to str at read time
df_month_ele = load_newest_excel(month_ele, header_row=1, dtype={'EAN odběrného místa': str})
df_month_plyn = load_newest_excel(month_plyn, header_row=2, dtype={'EIC odběrného místa': str})
df_alltime_ele = load_newest_excel(alltime_ele, header_row=1, dtype={'EAN odběrného místa': str})
df_alltime_plyn = load_newest_excel(alltime_plyn, header_row=2, dtype={'EIC odběrného místa': str})
df_rozsireny = load_newest_excel(rozsireny, header_row=1)
df_ceny_chytil = pd.read_excel(ceny_chytil, header=0)

df_month_ele = df_month_ele[[
    'Název zákazníka',
    'Číslo dokladu', 
    'Datum vystavení',
    'Zdanitelné plnění',
    'Číslo\nsmlouvy',
    'EAN odběrného místa', 
    'Obor', 
    'Segment / Kategorie SÚ',
    'Zúčtování od', 
    'Zúčtování do', 
    'Produkt', 
    'Celková spotřeba v MWh']]

df_alltime_ele = df_alltime_ele[[
    'Název zákazníka',
    'Číslo dokladu', 
    'Datum vystavení',
    'Zdanitelné plnění',
    'Číslo\nsmlouvy',
    'EAN odběrného místa', 
    'Obor', 
    'Segment / Kategorie SÚ',
    'Zúčtování od', 
    'Zúčtování do', 
    'Produkt', 
    'Celková spotřeba v MWh']]

df_month_plyn = df_month_plyn[[
    'Název zákazníka',
    'Číslo dokladu',
    'Datum vystavení',
    'Zdanitelné plnění', 
    'Číslo smlouvy', 
    'EIC odběrného místa',
    'Obor',
    'Segment / Kategorie SÚ', 
    'Zúčtování od', 
    'Zúčtování do',
    'Produkt', 
    'Spotřeba MWh']]

df_alltime_plyn = df_alltime_plyn[[
    'Název zákazníka',
    'Číslo dokladu',
    'Datum vyrovnání', 
    'Datum vystavení',
    'Zdanitelné plnění', 
    'Číslo smlouvy', 
    'EIC odběrného místa',
    'Obor',
    'Segment / Kategorie SÚ', 
    'Zúčtování od', 
    'Zúčtování do',
    'Produkt', 
    'Spotřeba MWh']]

df_rozsireny = df_rozsireny[[
    'Jméno',
    'Příjmení',
    'Název firmy', 
    'Číslo smlouvy', 
    'Komoditní sazba - VT/ZP (MWh)'
]]

# přejmenování sloupců, aby matchovali mezi sebou
df_month_ele = df_month_ele.rename(columns={
    'Číslo\nsmlouvy': 'Číslo smlouvy', 
    'EAN odběrného místa': 'EAN/EIC',
    'Celková spotřeba v MWh': 'Spotřeba MWh'
})

df_alltime_ele = df_alltime_ele.rename(columns={
    'Číslo\nsmlouvy': 'Číslo smlouvy', 
    'EAN odběrného místa': 'EAN/EIC',
    'Celková spotřeba v MWh': 'Spotřeba MWh'
})

df_month_plyn = df_month_plyn.rename(columns={
   'EIC odběrného místa': 'EAN/EIC' 
})

df_alltime_plyn = df_alltime_plyn.rename(columns={
   'EIC odběrného místa': 'EAN/EIC', 
})

df_month_ele['Datum vystavení'] = pd.to_datetime(df_month_ele['Datum vystavení'])
df_alltime_ele['Číslo smlouvy'] = df_alltime_ele['Číslo smlouvy'].astype(str)

# filtrace řádků tak, aby se daly použít jen ty, kde jsou reálné faktury
df_alltime_plyn = df_alltime_plyn[(df_alltime_plyn['Název zákazníka'].isnull()) & (df_alltime_plyn['Číslo dokladu'].notna())]
df_alltime_ele = df_alltime_ele[(df_alltime_ele['Název zákazníka'].isnull()) & (df_alltime_ele['Číslo dokladu'].notna())]
df_month_ele = df_month_ele[(df_month_ele['Název zákazníka'].isnull()) & (df_month_ele['Číslo dokladu'].notna())]
df_month_plyn = df_month_plyn[(df_month_plyn['Název zákazníka'].isnull()) & (df_month_plyn['Číslo dokladu'].notna())]
      
# spojeni tabulek 
df_pocitabulka = pd.concat([df_month_ele, df_month_plyn])
df_alltimetabulka= pd.concat([df_alltime_ele, df_alltime_plyn])

# vytvoření tabulky minusu
df_minusy = df_pocitabulka.copy()
df_minusy['Finální provize'] = df_minusy['Produkt'].map({
    'EDEN VIP (6080 obchodníci) | PevnáCena':-500, 
    'EDEN PREMIUM (6080 obchodníci) | PevnáCena':-2000,
    'EDEN SUPER (6080 obchodníci) | PevnáCena:':-1000,
    'EDEN (6080 obchodníci) | PevnáCena':-2500
}).fillna(0)


# převod datových sloupců do správného formátu a následný datedif
df_pocitabulka['Zúčtování od'] = pd.to_datetime(df_pocitabulka['Zúčtování od'])
df_pocitabulka['Zúčtování do'] = pd.to_datetime(df_pocitabulka['Zúčtování do'])
df_pocitabulka['Datum vystavení'] = pd.to_datetime(df_pocitabulka['Datum vystavení'])
df_pocitabulka['Spotřeba MWh'] = df_pocitabulka['Spotřeba MWh'].astype(float)
df_ceny_chytil['období'] = pd.to_datetime(df_ceny_chytil['období'])
df_pocitabulka['datedif'] = (df_pocitabulka['Zúčtování do'] - df_pocitabulka['Zúčtování od']).dt.days / 30.44
df_alltimetabulka['Datum vystavení'] = pd.to_datetime(df_alltimetabulka['Datum vystavení'])



# dropping duplicates in all time tabulka tak, aby tam byla jen první faktura
df_alltimetabulka = df_alltimetabulka.sort_values('Datum vystavení')
df_alltimetabulka = df_alltimetabulka.drop_duplicates(subset=['Číslo smlouvy'], keep='first')
df_alltimetabulka = df_alltimetabulka.rename(columns={
    'Datum vystavení': 'Datum vystavení první faktury'
})

# Zaokrouhlení datedif sloupce a následné vytvoření řádků podle datedif sloupce 
df_pocitabulka = df_pocitabulka.reset_index(drop=True)
df_pocitabulka['datedif'] = df_pocitabulka['datedif'].round().astype(int)
df_pocitabulka = df_pocitabulka.loc[df_pocitabulka.index.repeat(df_pocitabulka['datedif'])].reset_index(drop=True)

df_pocitabulka['pocitadlo_mesicu'] = df_pocitabulka.groupby('Číslo dokladu').cumcount()
df_pocitabulka['měsíc_fakturace'] = df_pocitabulka.apply(
    lambda row: (row['Zúčtování od'].replace(day=1) + relativedelta(months=row['pocitadlo_mesicu'])), axis=1
)
# napojení ceny z ceny_chytil
df_pocitabulka = df_pocitabulka.merge(
    df_ceny_chytil[['Value', 'období', 'Attribute']], 
    left_on=['měsíc_fakturace', 'Obor'],
    right_on=['období', 'Attribute'],
    how='left'
)
# přejmenování sloupce s cenou
df_pocitabulka = df_pocitabulka.rename(columns={
    'Value': 'Cena v měsíci fakturace'})

# napojení tabulky s prvními fakturami
df_pocitabulka = df_pocitabulka.merge(
    df_alltimetabulka[['Číslo smlouvy', 'Datum vystavení první faktury']], 
    left_on=['Číslo smlouvy'], 
    right_on=['Číslo smlouvy'], 
    how= 'left'
)

df_minusy = df_minusy.merge(
    df_alltimetabulka[['Číslo smlouvy', 'Datum vystavení první faktury']], 
    left_on=['Číslo smlouvy'], 
    right_on=['Číslo smlouvy'], 
    how='left'
)
df_minusy = df_minusy[df_minusy['Datum vystavení'] == df_minusy['Datum vystavení první faktury']]
df_rozsireny['Zákazník'] = (
    df_rozsireny['Jméno'].fillna('') + ' ' + 
    df_rozsireny['Příjmení'].fillna('') + ' ' + 
    df_rozsireny['Název firmy'].fillna('')
)


df_pocitabulka = df_pocitabulka.merge(
    df_rozsireny[['Číslo smlouvy', 'Zákazník', 'Komoditní sazba - VT/ZP (MWh)']],
    left_on='Číslo smlouvy', 
    right_on='Číslo smlouvy', 
    how='left'
)

df_pocitabulka['Poměr MWh na měsíc fakturace'] = df_pocitabulka['Spotřeba MWh']/df_pocitabulka['datedif']
df_pocitabulka['Finální provize'] = df_pocitabulka['Poměr MWh na měsíc fakturace']*(df_pocitabulka['Komoditní sazba - VT/ZP (MWh)']-df_pocitabulka['Cena v měsíci fakturace'])
df_final_tabulka = pd.concat([df_pocitabulka, df_minusy])
df_final_tabulka = df_final_tabulka.drop(columns=[
    'pocitadlo_mesicu', 
    'Poměr MWh na měsíc fakturace', 
    'Název zákazníka', 
    'období', 
    'Attribute'
])
df_final_tabulka = df_final_tabulka[[
    'Zákazník',
    'Číslo dokladu',
    'Číslo smlouvy',
    'EAN/EIC',
    'Obor',
    'Datum vystavení',
    'měsíc_fakturace',
    'Cena v měsíci fakturace',
    'Komoditní sazba - VT/ZP (MWh)',
    'Finální provize', 
    'Produkt', 
    'Spotřeba MWh'
]]
df_final_tabulka = df_final_tabulka.sort_values(
    by=['Číslo dokladu', 'měsíc_fakturace', 'Finální provize'],
    ascending=[True, True, True]
)
# Round Finální provize to 2 decimal places
df_final_tabulka['Finální provize'] = df_final_tabulka['Finální provize'].round(2)

# First save normally
df_final_tabulka.to_excel(final_path, index=False, startrow=2)

# Then open and add table formatting
wb = load_workbook(final_path)
ws = wb.active

# Force EAN/EIC column to text format so Excel doesn't show scientific notation
header_row = [cell.value for cell in ws[3]]
ean_col_idx = header_row.index('EAN/EIC') + 1  # 1-based

for row in range(2, ws.max_row + 1):
    cell = ws.cell(row=row, column=ean_col_idx)
    cell.number_format = numbers.FORMAT_TEXT
    cell.value = str(cell.value) if cell.value is not None else None

# Define table range (A1 to last column and row)
last_col = chr(64 + ws.max_column)  # converts number to letter (e.g. 1→A, 2→B)
table_range = f"A3:{last_col}{ws.max_row}"
ws['A1'] = '6080 MILAN CHYTIL'
ws['A2'] = 'chyta10@seznam.cz'
ws['H1'] ='K FAKTURACI CELKEM'
ws['H2'] = 'K FAKTURACI CELKEM'
ws['I1'] = 'PROVIZE'
ws['I2'] = 'PROLONG'
ws['J1'] = f'=SUM(J3:J20000)'
from openpyxl.styles import Font

blue_bold = Font(bold=True, color="0070C0", size=22)

for row in ws.iter_rows(min_row=1, max_row=2, max_col=ws.max_column):
    for cell in row:
        cell.font = blue_bold

# Create table
table = Table(displayName="Provize", ref=table_range)
style = TableStyleInfo(name="TableStyleMedium2", showRowStripes=True)
table.tableStyleInfo = style

ws.add_table(table)
wb.save(final_path)

Loading: zs_FakturovanPolozkyElektriny (8).xlsx
Loading: zs_FakturovanePolozkyPlyn (6).xlsx
Loading: zs_FakturovanPolozkyElektriny_ChytilAlltime.xlsx
Loading: zs_FakturovanePolozkyPlyn_ChytilAlltime.xlsx
Loading: 20260318_DetailSmluvZPaEERozsireny.xlsx


C:\Users\samuel.ackah\AppData\Local\Temp\ipykernel_29120\2050282461.py:135: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_month_ele['Datum vystavení'] = pd.to_datetime(df_month_ele['Datum vystavení'])
C:\Users\samuel.ackah\AppData\Local\Temp\ipykernel_29120\2050282461.py:165: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_alltimetabulka['Datum vystavení'] = pd.to_datetime(df_alltimetabulka['Datum vystavení'])
